In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "chatdb/natural-sql-7b"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# --- 修改後的 GPU 加速設定 ---
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",          # ⭐️ 關鍵改動：讓 AI 自動分配 GPU 和 CPU 記憶體
    torch_dtype=torch.float16,  # ⭐️ 關鍵改動：使用半精度 (節省一半 VRAM)
    low_cpu_mem_usage=True
)

print("模型載入完成！正在使用的裝置:", model.device)

c:\Users\user\Documents\llm_tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\user\Documents\llm_tutorial\.venv\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\user\Documents\llm_tutorial\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\Users\user\Documents\llm_tutorial\.ve

模型載入完成！正在使用的裝置: cuda:0


In [ ]:
import sqlite3
import pandas as pd
import gradio as gr

# ==========================================
# 1. 準備資料庫與 Schema (沿用之前的設定)
# ==========================================
schema = """
CREATE TABLE samples (id INTEGER PRIMARY KEY, sample_external_id TEXT, cancer_type TEXT, collection_date DATE);
CREATE TABLE plate (id INTEGER PRIMARY KEY, plate_index TEXT);
CREATE TABLE wells (id INTEGER PRIMARY KEY, well_name TEXT, sample_id INTEGER, plate_id TEXT, dapi_count INTEGER);
CREATE TABLE models (id INTEGER PRIMARY KEY, model_name TEXT);
CREATE TABLE well_predictions (well_id INTEGER, model_id INTEGER, positive_count INTEGER);
CREATE TABLE cells (well_id INTEGER, r_net_mean REAL);
"""

# 建立模擬資料庫 (每次執行重置)
conn = sqlite3.connect(':memory:', check_same_thread=False)
cursor = conn.cursor()
cursor.executescript(schema)
cursor.executescript("""
INSERT INTO samples VALUES (1, 'S_LUNG_01', 'Lung', '2023-01-10'), (2, 'S_BREAST_01', 'Breast', '2023-01-12'), (3, 'S_LUNG_02', 'Lung', '2023-02-20');
INSERT INTO plate VALUES (1, 'P001');
INSERT INTO wells VALUES (1, 'A01', 1, 'P001', 1200), (2, 'B01', 2, 'P001', 1500);
INSERT INTO models VALUES (1, 'v2_model');
INSERT INTO well_predictions VALUES (1, 1, 55);
INSERT INTO cells VALUES (1, 10.0), (1, 20.0), (1, 60.0);
""")
conn.commit()

# ==========================================
# 2. 定義核心函數 (GPU 加速版 🚀)
# ==========================================
def process_question(question):
    print(f"收到問題: {question} (GPU 高速生成中...)")
    
    # 檢查模型是否載入
    if 'model' not in globals() or 'tokenizer' not in globals():
        return "❌ 錯誤：找不到 model 變數，請確認上一步模型已載入成功。", "無法執行"

    # --- A. 生成 SQL ---
    prompt = f"""
    ### Task 
    Generate a SQL query to answer the following question: `{question}` 
    
    ### PostgreSQL Database Schema 
    The query will run on a database with the following schema: 
    ```sql
    {schema}
    ```
    
    ### Answer 
    Here is the SQL query that answers the question: `{question}` 
    ```sql
    """
    
    try:
        # ⭐️ 關鍵修改：加上 .to("cuda") 將輸入傳送給顯卡
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
        )
        
        raw_output = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        if "```sql" in raw_output:
            sql_query = raw_output.split("```sql")[-1].strip().replace("```", "")
        else:
            sql_query = raw_output
            
    except Exception as e:
        return f"❌ 生成失敗: {e}", "Error"

    # --- B. 執行 SQL (驗證階段) ---
    try:
        # 語法轉換 (PostgreSQL -> SQLite)
        fixed_sql = sql_query.replace("::date", "").replace("ILIKE", "LIKE").replace("ilike", "LIKE")
        
        # 修正 JOIN 邏輯 (針對之前的 plate_id 問題)
        # 如果 AI 寫錯成 plate.id，我們嘗試幫它修正為 plate.plate_index (這是一個簡單的自動修復邏輯)
        if "ON wells.plate_id = plate.id" in fixed_sql:
            fixed_sql = fixed_sql.replace("ON wells.plate_id = plate.id", "ON wells.plate_id = plate.plate_index")

        df = pd.read_sql_query(fixed_sql, conn)
        
        if df.empty:
            return sql_query, "⚠️ 執行成功，但查無資料 (可能條件不符)"
        else:
            return sql_query, df
            
    except Exception as e:
        return sql_query, f"❌ SQL 執行錯誤: {str(e)}"

# ==========================================
# 3. 啟動極速版介面
# ==========================================
demo = gr.Interface(
    fn=process_question, 
    inputs=gr.Textbox(lines=2, placeholder="例如: Find all Lung cancer samples...", label="請輸入問題"),
    outputs=[
        gr.Code(language="sql", label="1. AI 生成的 SQL (GPU Accelerated)"),
        gr.Dataframe(label="2. 模擬資料庫執行結果")
    ], 
    title="🧬 生物資訊 AI 助理 (🚀 NVIDIA GPU 加速版)",
    description="現在模型運行在 CUDA 上！請體驗一下生成速度的差別。",
    examples=[
        ["Find all samples with cancer_type 'Lung'"],
        ["Calculate the total dapi_count for plate 'P001'"],
        ["Find the average r_net_mean for cells in well 'A01'"]
    ]
)

demo.launch()

In [2]:
import sqlite3
import pandas as pd
import gradio as gr
import os

# ==========================================
# 1. 設定資料庫來源 (請修改這裡！)
# ==========================================
# 如果你有自己的 db 檔，請把檔名改成你的，例如: real_db_path = "my_experiment.db"
real_db_path = "C:/Users/user/Downloads/inwell_data.db"
# --- (以下這段只是為了演示，如果你沒有檔案，它會自動建立一個假的) ---
if not os.path.exists(real_db_path):
    print(f"⚠️ 找不到 {real_db_path}，正在建立一個測試用的資料庫...")
    conn = sqlite3.connect(real_db_path)
    cursor = conn.cursor()
    # 建立一些範例資料表
    cursor.execute("CREATE TABLE patients (id INTEGER PRIMARY KEY, name TEXT, age INTEGER, diagnosis TEXT, admission_date DATE)")
    cursor.execute("CREATE TABLE treatments (id INTEGER PRIMARY KEY, patient_id INTEGER, drug_name TEXT, dosage TEXT, start_date DATE)")
    # 塞入資料
    cursor.execute("INSERT INTO patients VALUES (1, 'Alice', 30, 'Lung Cancer', '2023-01-01'), (2, 'Bob', 45, 'Flu', '2023-02-15')")
    cursor.execute("INSERT INTO treatments VALUES (1, 1, 'Cisplatin', '50mg', '2023-01-02'), (2, 2, 'Tamiflu', '75mg', '2023-02-16')")
    conn.commit()
    conn.close()
    print("✅ 測試資料庫建立完成！")
# -----------------------------------------------------------

# ==========================================
# 2. 定義自動掃描函數
# ==========================================
def get_schema_from_db(db_path):
    """連線到 .db 檔，自動生成 CREATE TABLE schema"""
    if not os.path.exists(db_path):
        return f"Error: 找不到檔案 {db_path}"
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # 抓取所有表名
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
    tables = cursor.fetchall()
    
    schema_str = ""
    for table in tables:
        table_name = table[0]
        # 抓取欄位資訊
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        
        col_defs = []
        for col in columns:
            # col[1]=name, col[2]=type
            col_defs.append(f"    {col[1]} {col[2]}")
            
        schema_str += f"CREATE TABLE {table_name} (\n" + ",\n".join(col_defs) + "\n);\n"
    
    conn.close()
    return schema_str

# 自動抓取目前的 Schema
current_schema = get_schema_from_db(real_db_path)
print("--- 自動抓取到的 Schema ---")
print(current_schema)

# ==========================================
# 3. 定義處理函數 (生成 + 執行)
# ==========================================
def process_question(question):
    # 檢查模型是否載入
    if 'model' not in globals() or 'tokenizer' not in globals():
        return "❌ 錯誤：模型尚未載入，請先執行載入模型的 Cell。", "Error"

    print(f"收到問題: {question}")
    
    # A. 生成 SQL
    prompt = f"""
    ### Task 
    Generate a SQL query to answer the following question: `{question}` 
    
    ### Database Schema 
    The query will run on a database with the following schema: 
    ```sql
    {current_schema}
    ```
    
    ### Answer 
    Here is the SQL query that answers the question: `{question}` 
    ```sql
    """
    
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda") # GPU 加速
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=400, 
            do_sample=False
        )
        raw_output = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        if "```sql" in raw_output:
            sql_query = raw_output.split("```sql")[-1].strip().replace("```", "")
        else:
            sql_query = raw_output
            
    except Exception as e:
        return f"生成失敗: {e}", "Error"

    # B. 執行 SQL (直接連線真實 DB)
    try:
        conn = sqlite3.connect(real_db_path)
        # 簡單的語法修正
        fixed_sql = sql_query.replace("::date", "").replace("ILIKE", "LIKE").replace("ilike", "LIKE")
        
        df = pd.read_sql_query(fixed_sql, conn)
        conn.close()
        
        if df.empty:
            return sql_query, "⚠️ 查無資料"
        else:
            return sql_query, df
            
    except Exception as e:
        return sql_query, f"❌ 執行錯誤: {str(e)}"

# ==========================================
# 4. 啟動 Gradio
# ==========================================
demo = gr.Interface(
    fn=process_question, 
    inputs=gr.Textbox(label="請輸入問題"),
    outputs=[gr.Code(label="生成的 SQL"), gr.Dataframe(label="查詢結果")], 
    title="🧬 全自動資料庫 AI 助理",
    description=f"目前已連接資料庫：`{real_db_path}`\n\n系統已自動掃描並學習了其中的表格結構。",
    examples=[
        ["Find all patients older than 40"],
        ["Show me the treatments for patient Alice"],
        ["Count how many patients have 'Lung Cancer'"]
    ]
)

demo.launch()

--- 自動抓取到的 Schema ---
CREATE TABLE samples (
    id INTEGER,
    sample_external_id TEXT,
    site TEXT,
    birth_date DATE,
    collection_date DATE,
    sex TEXT,
    cancer_type TEXT,
    subtype TEXT,
    stage INT,
    treatment BOOLEAN
);
CREATE TABLE plates (
    id INTEGER,
    plate_index TEXT,
    plate_type TEXT
);
CREATE TABLE wells (
    id INTEGER,
    well_name TEXT,
    sample_id INTEGER,
    plate_id INTEGER,
    well_coordinate TEXT,
    culture_time TEXT,
    panel TEXT,
    marker TEXT,
    dapi_count INTEGER,
    created_at TIMESTAMP
);
CREATE TABLE cells (
    id INTEGER,
    well_id INTEGER,
    object_number INTEGER,
    area INTEGER,
    coord_x REAL,
    coord_y REAL,
    eccentricity REAL,
    b_mean REAL,
    g_mean REAL,
    r_mean REAL,
    t_mean REAL,
    r_net_mean REAL,
    g_net_mean REAL,
    b_net_mean REAL,
    t_net_mean REAL
);
CREATE TABLE models (
    id INTEGER,
    model_id INTEGER,
    model_name TEXT,
    note TEXT
);
CREATE TABLE model_pr

c:\Users\user\Documents\llm_tutorial\.venv\Lib\site-packages\transformers\generation\utils.py:1473: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


收到問題: Count how many positive cells are there
